In [2]:
import h5py
import numpy as np
import pickle
from pathlib import Path
# One-time conversion from your pickle to HDF5
def convert_pickle_to_hdf5(pickle_path, hdf5_path):
    with open(pickle_path, 'rb') as f:
        data_list = pickle.load(f)
    
    with h5py.File(hdf5_path, 'w') as f:
        # Store each dictionary field as separate datasets
        for key in data_list[0].keys():
            if key == 'img_path':
                continue
            # Convert to numpy, keep float32, stack along first axis
            values = [item[key] for item in data_list]  # already float32
            f.create_dataset(key, data=np.stack(values, axis=0), dtype='float32')

In [6]:
train_path = Path("../data") / 'IMC-sample' / 'train.pkl'
train_hdf5_path = Path("../data") / 'IMC-sample' / 'train.h5'

test_path = Path("../data") / 'IMC-sample' / 'test.pkl'
test_hdf5_path = Path("../data") / 'IMC-sample' / 'test.h5'

In [7]:
convert_pickle_to_hdf5(train_path, train_hdf5_path)

In [8]:
convert_pickle_to_hdf5(test_path, test_hdf5_path)

# New data structure

In [ ]:
import pickle
import h5py
import numpy as np
from pathlib import Path

def convert_pickle_to_hdf5(pickle_path, hdf5_path):
    with open(pickle_path, 'rb') as f:
        data_dict = pickle.load(f)

    with h5py.File(hdf5_path, 'w') as f:
        all_embeddings = []
        all_metadata = []
        all_positions = []
        all_paths = []
        region_keys_ref = None  # store once

        for path_name, records in data_dict.items():
            for rec in records:
                # detect region keys dynamically (e.g. r0_f, r0_nf, ...)
                region_keys = [k for k in rec.keys() if k != 'position']
                region_keys.sort()  # optional, ensure consistent order

                if region_keys_ref is None:
                    region_keys_ref = region_keys

                # stack embeddings (8, D)
                embeddings = np.stack(
                    [np.asarray(rec[r]['embedding'], dtype=np.float32) for r in region_keys],
                    axis=0
                )

                # stack metadata (8, 3)
                metadata = np.stack(
                    [[rec[r]['loss'], rec[r]['mae'], rec[r]['rotation_error']] for r in region_keys],
                    axis=0
                ).astype(np.float32)

                # extract position (x_start, x_stop, y_start, y_stop)
                pos = rec['position']
                if isinstance(pos[0], slice):
                    pos_array = np.array([
                        pos[0].start, pos[0].stop,
                        pos[1].start, pos[1].stop
                    ], dtype=np.float32)
                else:
                    pos_array = np.array(pos, dtype=np.float32)

                all_embeddings.append(embeddings)
                all_metadata.append(metadata)
                all_positions.append(pos_array)
                all_paths.append(path_name)

        # Convert to arrays
        all_embeddings = np.stack(all_embeddings, axis=0)  # (N, 8, D)
        all_metadata = np.stack(all_metadata, axis=0)      # (N, 8, 3)
        all_positions = np.stack(all_positions, axis=0)    # (N, 4)

        # Save datasets
        f.create_dataset('embeddings', data=all_embeddings, dtype='float32')
        f.create_dataset('metadata', data=all_metadata, dtype='float32')
        f.create_dataset('positions', data=all_positions, dtype='float32')

        dt = h5py.special_dtype(vlen=str)
        f.create_dataset('paths', data=np.array(all_paths, dtype=object), dtype=dt)

        print(f"✅ Saved {len(all_embeddings)} samples to {hdf5_path}")
        print(f"Region keys: {region_keys_ref}")


In [ ]:
train_path = Path("/raid/tnocon/data") / 'IMC' / 'nsclc2_panel1_test.pkl'
train_hdf5_path = Path("/raid/tnocon/data") / 'IMC' / 'nsclc2_panel1_test.h5'

In [ ]:
convert_pickle_to_hdf5(train_path, train_hdf5_path)

In [ ]:
# train_hdf5_path = Path("/raid/tnocon/data") / 'IMC' / 'train.h5'

d = PickleDataset(train_hdf5_path)

In [ ]:
import math
from pathlib import Path
import h5py
import numpy as np

import numpy as np
import h5py

def _safe_chunks(sch, total_len, chunk_rows):
    # Limits
    max_elems = 2_147_483_647           # < 2^31 elements
    max_bytes = 2_147_483_647           # < 2^31 bytes (safe across HDF5 versions)

    # Try reusing source chunks only if they satisfy both limits
    if sch["chunks"] is not None:
        src = sch["chunks"]
        elem_count = int(np.prod(src))
        if sch["is_string"]:
            bytes_count = elem_count  # varlen strings: can only check elements
        else:
            bytes_count = elem_count * np.dtype(sch["dtype"]).itemsize
        if elem_count <= max_elems and bytes_count <= max_bytes:
            return src

    # Compute safe chunks along axis 0
    if sch["is_string"]:
        tail = sch["shape_tail"]
        rows = max(1, min(chunk_rows, total_len, max_elems))
        return (rows,) + tail

    tail = sch["shape_tail"]
    tail_elems = int(np.prod(tail)) if len(tail) else 1
    itemsize = np.dtype(sch["dtype"]).itemsize

    # Bound rows by element and byte limits
    max_rows_by_elems = max(1, max_elems // max(1, tail_elems))
    max_rows_by_bytes = max(1, max_bytes // max(1, tail_elems * itemsize))
    rows = max(1, min(chunk_rows, total_len, max_rows_by_elems, max_rows_by_bytes))
    return (rows,) + tail

def _copy_attrs(src_obj, dst_obj):
    for k, v in src_obj.attrs.items():
        dst_obj.attrs[k] = v

def merge_h5_axis0(input_paths, output_path,
                   dataset_names=("embeddings", "metadata", "positions", "paths"),
                   chunk_rows=8192):
    if len(input_paths) < 2:
        raise ValueError("Provide at least two input files.")
    input_paths = [str(p) for p in input_paths]

    # Open first to define schema
    with h5py.File(input_paths[0], "r") as f0:
        # Validate all required datasets exist
        for name in dataset_names:
            if name not in f0:
                raise KeyError(f"Dataset '{name}' not found in {input_paths[0]}")
        # Collect trailing shapes and dtypes from first file
        schema = {}
        for name in dataset_names:
            d0 = f0[name]
            shape0 = d0.shape
            if len(shape0) == 0:
                raise ValueError(f"Dataset '{name}' must be at least 1D.")
            schema[name] = {
                "dtype": d0.dtype,
                "shape_tail": shape0[1:],
                "is_string": h5py.check_string_dtype(d0.dtype) is not None,
                "compression": d0.compression,
                "compression_opts": d0.compression_opts,
                "shuffle": d0.shuffle,
                "fletcher32": d0.fletcher32,
                "chunks": d0.chunks,
            }

    # Compute total rows N across files; validate compatibility
    per_file_lengths = []
    for p in input_paths:
        with h5py.File(p, "r") as f:
            lens = []
            for name in dataset_names:
                ds = f[name]
                # trailing shape and dtype must match (except strings we treat by check_string_dtype)
                tail = ds.shape[1:]
                if tail != schema[name]["shape_tail"]:
                    raise ValueError(f"Trailing shape mismatch for '{name}' in {p}: "
                                     f"{tail} vs {schema[name]['shape_tail']}")
                if schema[name]["is_string"]:
                    if h5py.check_string_dtype(ds.dtype) is None:
                        raise ValueError(f"'{name}' expected string dtype, but found {ds.dtype} in {p}")
                else:
                    if ds.dtype != schema[name]["dtype"]:
                        raise ValueError(f"Dtype mismatch for '{name}' in {p}: {ds.dtype} vs {schema[name]['dtype']}")
                lens.append(ds.shape[0])
            # ensure all datasets in file have same leading length
            if len(set(lens)) != 1:
                raise ValueError(f"Datasets have different lengths in {p}: {lens}")
            per_file_lengths.append(lens[0])

    total_len = int(np.sum(per_file_lengths))

    with h5py.File(output_path, "w") as fout:
        # Create output datasets
        # inside merge_h5_axis0(...)
        outs = {}
        for name in dataset_names:
            sch = schema[name]
            out_shape = (total_len,) + sch["shape_tail"]
            chunks = _safe_chunks(sch, total_len, chunk_rows)

            if sch["is_string"]:
                # Avoid compression/shuffle/fletcher32 on variable-length strings
                ds_out = fout.create_dataset(
                    name, shape=out_shape, dtype=h5py.string_dtype(encoding="utf-8"),
                    chunks=chunks,
                )
            else:
                ds_out = fout.create_dataset(
                    name, shape=out_shape, dtype=sch["dtype"],
                    chunks=chunks,
                    compression=sch["compression"], compression_opts=sch["compression_opts"],
                    shuffle=sch["shuffle"], fletcher32=sch["fletcher32"],
                )
            outs[name] = ds_out

        # Optionally copy file-level attrs from the first file
        with h5py.File(input_paths[0], "r") as f0:
            _copy_attrs(f0, fout)
            for name in dataset_names:
                _copy_attrs(f0[name], outs[name])

        # Streamed copy
        write_pos = 0
        for p, n_rows in zip(input_paths, per_file_lengths):
            with h5py.File(p, "r") as fin:
                for name in dataset_names:
                    src = fin[name]
                    dst = outs[name]
                    # pick a row block size
                    rows = (src.chunks[0] if src.chunks else chunk_rows)
                    # ensure at least 1
                    rows = max(1, rows)
                    steps = math.ceil(n_rows / rows)
                    for i in range(steps):
                        s = i * rows
                        e = min((i + 1) * rows, n_rows)
                        dst[write_pos + s: write_pos + e] = src[s:e]
            write_pos += n_rows

    print(f"✅ Merged {len(input_paths)} files into {output_path} with total rows={total_len}")

In [ ]:
p1 = Path("/raid/tnocon/data") / 'IMC' / 'nsclc2_panel1_train_1st_quarter.h5'
p2 = Path("/raid/tnocon/data") / 'IMC' / 'nsclc2_panel1_train_2nd_quarter.h5'
p3 = Path("/raid/tnocon/data") / 'IMC' / 'nsclc2_panel1_train_2nd_half.h5'

merge_h5_axis0([p1, p2, p3], Path("/raid/tnocon/data") / 'IMC' / 'nsclc2_panel1_train.h5')

In [ ]:
def merge_large_hdf5(file1, file2, out_file):
    """Backward-compatible wrapper that merges along axis 0 using merge_h5_axis0."""
    return merge_h5_axis0([file1, file2], out_file)

In [ ]:
merge_large_hdf5(p1, p2, Path("/raid/tnocon/data") / 'IMC' / 'nsclc2_panel1_train.h5')

In [2]:
import pandas as pd


test = pd.read_csv("~/Downloads/Liked_Songs.csv")